# nb03 patch generation

LunarAI — SIH 26166 notebook. Generated by build_notebooks.py.

## 1. Objective
Generate lunar patches from all datasets at **256x256 with stride 128**, attach metadata
(Patch_ID, Source_Image, Dataset_Name, Latitude, Longitude, Sun_Angle, Elevation,
Patch_Path) and produce `patch_index.csv` plus the SQLite patch table.

## 2. Dataset Loading

In [1]:
import sys
from pathlib import Path

def _find_project_root() -> Path:
    # anchor at the directory that actually contains the project markers,
    # independent of the kernel's working directory
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "lunarai_lib").is_dir() and (cand / "configs").is_dir():
            return cand
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from lunarai_lib.config import load_config
from lunarai_lib import metadata as md
from lunarai_lib import patching
from lunarai_lib.io_utils import save_json
from lunarai_lib.db import Database
import pandas as pd
import numpy as np

cfg = load_config()
records_by_sensor = md.discover_all(cfg.DATA_ROOT)
all_records = [r for recs in records_by_sensor.values() for r in recs]
elev_lookup = patching.build_elevation_lookup(cfg.DATA_ROOT / "ElevationProfile.csv")
print(f"{len(all_records)} products | elevation bins: {len(elev_lookup)}")

15 products | elevation bins: 30


## 3. Configuration

In [2]:
PATCH_SIZE = cfg.PATCH_SIZE          # 256
STRIDE = cfg.STRIDE                  # 128
MAX_PER_IMAGE = cfg.MAX_PATCHES_PER_IMAGE
MIN_STD = cfg.MIN_STD
print(f"patch {PATCH_SIZE}x{PATCH_SIZE} stride {STRIDE} cap {MAX_PER_IMAGE}/image")

patch 256x256 stride 128 cap 150/image


## 4. Implementation

In [3]:
all_metas = []
for r in all_records:
    metas = patching.generate_patches_for_record(
        r, cfg.PATCHES_DIR,
        patch_size=PATCH_SIZE, stride=STRIDE, max_patches=MAX_PER_IMAGE,
        min_std=MIN_STD, resize_long_side=cfg.RESIZE_LONG_SIDE,
        elev_lookup=elev_lookup,
        processed_dir=cfg.PROCESSED_DIR,
        preprocess_fn=None,   # patches inherit NB02-enhanced imagery; raw patches kept here
        sensor_tag=r.sensor.replace(" ", ""),
    )
    if r.notes:
        print(f"  [{r.sensor:7s}] {r.image_id[:40]:40s} -> {len(metas):3d} patches  ({r.notes.strip()})")
    else:
        print(f"  [{r.sensor:7s}] {r.image_id[:40]:40s} -> {len(metas):3d} patches")
    all_metas.extend(metas)

index_path = patching.write_patch_index(all_metas, cfg.PATCHES_DIR / "patch_index.csv")
df = pd.read_csv(index_path)
print(f"\nTotal patches: {len(df)}")
print(df.groupby("dataset_name").size())

  [OHRC   ] ch2_ohr_ncp_20241115T1326321339_d_img_d1 ->  11 patches


  [OHRC   ] ch2_ohr_ncp_20241115T1525004388_d_img_d1 ->  11 patches


  [OHRC   ] ch2_ohr_ncp_20251010T0942085687_d_img_d1 ->  13 patches


  [TMC-2  ] ch2_tmc_nca_20240124T0838058366_d_img_d1 -> 107 patches


  [TMC-2  ] ch2_tmc_ncf_20211122T1925079181_d_img_d1 ->  97 patches


...

  [TMC-2  ] ch2_tmc_ncf_20260701T0716265753_d_img_d1 ->  53 patches


  [TMC-2  ] ch2_tmc_ncn_20211122T1925079214_d_img_d1 ->  95 patches


  [TMC-2  ] ch2_tmc_ncn_20211123T0119537815_d_img_d1 ->  94 patches


  [IIRS   ] ch2_iir_nri_20210126T1349194455_d_img_d3 -> 132 patches  (patches from browse PNG fallback;)
  [IIRS   ] ch2_iir_nri_20210126T1545362143_d_img_d3 ->   0 patches
  [IIRS   ] ch2_iir_nri_20210126T1743334783_d_img_d3 ->   0 patches


  [IIRS   ] ch2_iir_nri_20240124T1430209615_d_img_d1 ->  97 patches  (patches from browse PNG fallback;)
  [LRO NAC] quickmap-lroc.png                        ->  36 patches

Total patches: 940
dataset_name
IIRS       229
LRO NAC     36
OHRC        35
TMC-2      640
dtype: int64


### SQLite persistence (Backend Schema: `patches`, `images` tables)

In [4]:
db = Database(cfg.DATABASE_DIR / "lunarai.db")
db.executemany(
    "INSERT OR REPLACE INTO images(image_id, filename, mission, sensor, latitude, longitude,"
    " resolution, sun_angle, acquisition_date) VALUES (?,?,?,?,?,?,?,?,?)",
    [(r.image_id, Path(r.path).name, r.mission, r.sensor,
      None if np.isnan(r.latitude) else float(r.latitude),
      None if np.isnan(r.longitude) else float(r.longitude),
      None if np.isnan(r.pixel_resolution) else float(r.pixel_resolution),
      None if np.isnan(r.sun_elevation) else float(r.sun_elevation),
      r.acquisition_date) for r in all_records])
def _n(v):
    return None if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)

db.executemany(
    "INSERT OR REPLACE INTO patches(patch_id, image_id, patch_path, latitude, longitude,"
    " patch_size, sensor) VALUES (?,?,?,?,?,?,?)",
    [(m.patch_id, m.source_image, m.patch_path, _n(m.latitude), _n(m.longitude),
      m.width, m.dataset_name) for m in all_metas])
print("DB rows:", db.query("SELECT sensor, COUNT(*) FROM patches GROUP BY sensor"))

DB rows: [('IIRS', 229), ('LRO NAC', 36), ('OHRC', 35), ('TMC-2', 640)]


## 5. Visualization — patch mosaic per sensor

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sensor in zip(axes, ["OHRC", "TMC-2", "IIRS", "LRO NAC"]):
    sub = df[df.dataset_name == sensor]
    if len(sub):
        img = cv2.imread(sub.patch_path.iloc[len(sub) // 2], cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{sensor}\n{len(sub)} patches")
    else:
        ax.set_title(f"{sensor}\n0 patches")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(cfg.VISUALIZATIONS_DIR / "patch_mosaic.png", dpi=120)
plt.show()

C:\Users\girid\AppData\Local\Temp\ipykernel_34408\3333054387.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Metrics

In [6]:
metrics = {
    "total_patches": int(len(df)),
    "per_sensor": df.groupby("dataset_name").size().to_dict(),
    "patch_size": PATCH_SIZE, "stride": STRIDE,
    "with_latitude": int(df.latitude.notna().sum()),
    "with_sun_angle": int(df.sun_angle.notna().sum()),
    "with_elevation": int(df.elevation.notna().sum()),
}
print(metrics)

{'total_patches': 940, 'per_sensor': {'IIRS': 229, 'LRO NAC': 36, 'OHRC': 35, 'TMC-2': 640}, 'patch_size': 256, 'stride': 128, 'with_latitude': 904, 'with_sun_angle': 772, 'with_elevation': 0}


## 7. Saved Outputs
- `outputs/patches/<sensor>/<patch_id>.png` (256x256)
- `outputs/patches/patch_index.csv`
- `database/lunarai.db` (patches + images tables populated)

In [7]:
save_json(metrics, cfg.METRICS_DIR / "patch_generation_report.json")

WindowsPath('D:/lunar ai/outputs/metrics/patch_generation_report.json')

## 8. Error Handling
- Records with missing binaries fall back to browse PNGs (flagged in notes).
- Patches with std < MIN_STD (blank/over-exposed) are rejected.
- Oversized grids are subsampled evenly (linspace) to respect the per-image cap.

## 9. Explanation
Patches tile the image on a 128 grid; edges get a final aligned row/column so full
coverage is kept. Geographic metadata is interpolated from PDS4 corner coordinates by
patch-center position; sun angle is inherited from the product label; elevation comes
from a dilated nearest-degree lookup into the LOLA DEM profile.

## 10. Next-Step Integration
Notebook 04 consumes `patch_index.csv` to build cross-sensor geo-triplets and train LunaDNA.